In [1]:
import os
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats
from generate_data import generate_pesapap_data

# Step 1: Generate Raw Data
generate_pesapap_data()

# Step 2: Build SQLite Database
conn = sqlite3.connect('pesapap.db')
users = pd.read_csv('users.csv')
txns = pd.read_csv('transactions.csv')
exp = pd.read_csv('experiment.csv')

users.to_sql('users', conn, if_exists='replace', index=False)
txns.to_sql('transactions', conn, if_exists='replace', index=False)
exp.to_sql('experiment', conn, if_exists='replace', index=False)

# SQL Queries Execution
Reading queries directly from `sql/queries.sql` using `pd.read_sql`.

In [2]:
with open('sql/queries.sql', 'r') as f:
    sql_file = f.read()

# Split individual queries
sql_queries = [q.strip() for q in sql_file.split(';') if q.strip()]

for idx, query in enumerate(sql_queries):
    print(f"=== Query {idx + 1} ===")
    res = pd.read_sql(query, conn)
    display(res)
    print("\n")

# KPI Scorecard & Metric Dictionary

| Metric Name | Definition | Calculation |
|---|---|---|
| **Total Transaction Value** | Sum of transaction amounts across the entire platform | `SUM(amount)` |
| **Number of Transactions** | Count of all logged transaction rows | `COUNT(txn_id)` |
| **Active Users** | Count of unique users making at least one transaction | `COUNT(DISTINCT user_id)` |
| **Average Transaction Value** | Mean monetary amount per transaction | `SUM(amount) / COUNT(txn_id)` |
| **Avg Txns per Active User** | Transaction frequency per engaged customer | `COUNT(txn_id) / COUNT(DISTINCT user_id)` |
| **MoM Active User Growth** | Percentage change in monthly active users | `(Current Active - Prev Active) / Prev Active * 100` |

In [3]:
total_val = txns['amount'].sum()
num_txns = len(txns)
active_users = txns['user_id'].nunique()
avg_txn_val = txns['amount'].mean()
avg_txns_per_user = num_txns / active_users

kpi_df = pd.DataFrame({
    'Metric': ['Total Value (KES)', 'Number of Txns', 'Active Users', 'Avg Txn Value (KES)', 'Avg Txns / Active User'],
    'Value': [f"{total_val:,.0f}", f"{num_txns:,}", f"{active_users:,}", f"{avg_txn_val:,.2f}", f"{avg_txns_per_user:.2f}"]
})
display(kpi_df)

# RFM Segmentation & Cohort Retention

In [4]:
# RFM Calculation
max_date = pd.to_datetime(txns['txn_date']).max()
txns['txn_dt'] = pd.to_datetime(txns['txn_date'])

rfm = txns.groupby('user_id').agg(
    Recency=('txn_dt', lambda x: (max_date - x.max()).days),
    Frequency=('txn_id', 'count'),
    Monetary=('amount', 'sum')
).reset_index()

# Categorize into RFM Tiers
rfm['R_score'] = pd.qcut(rfm['Recency'], 3, labels=['High', 'Med', 'Low'])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 3, labels=['Low', 'Med', 'High'])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 3, labels=['Low', 'Med', 'High'])

rfm_counts = rfm.groupby(['R_score', 'F_score', 'M_score'], observed=False).size().reset_index(name='user_count')
print("--- RFM SEGMENTS HEAD ---")
display(rfm_counts.head())
print("\n*Target Recommendation:* Focus retention campaigns on 'Low Recency / High Frequency / High Monetary' (At-Risk Champions) first to protect high-value revenue.")

In [5]:
# Cohort Retention Table
users['signup_dt'] = pd.to_datetime(users['signup_date'])
users['signup_cohort'] = users['signup_dt'].dt.to_period('M')
txns['txn_period'] = txns['txn_dt'].dt.to_period('M')

cohort_df = txns.merge(users[['user_id', 'signup_cohort']], on='user_id')
cohort_grouped = cohort_df.groupby(['signup_cohort', 'txn_period']).agg(active_users=('user_id', 'nunique')).reset_index()

# Period Number
cohort_grouped['cohort_index'] = (cohort_grouped['txn_period'] - cohort_grouped['signup_cohort']).apply(lambda x: x.n)
cohort_pivot = cohort_grouped.pivot(index='signup_cohort', columns='cohort_index', values='active_users')

# Retention Percentage
cohort_sizes = users.groupby('signup_cohort')['user_id'].nunique()
retention_matrix = cohort_pivot.divide(cohort_sizes, axis=0) * 100

print("--- COHORT RETENTION MATRIX (%) [Incomplete cells flagged as NaN] ---")
display(retention_matrix.round(1))

# Onboarding Experiment Analysis (A/B Test)

* **$H_0$ (Null Hypothesis):** The simplified sign-up screen (Group B) does not change conversion rate relative to the existing flow (Group A) ($p_A = p_B$).
* **$H_1$ (Alternative Hypothesis):** The simplified sign-up screen (Group B) increases conversion rate relative to Group A ($p_B > p_A$).

In [6]:
# Group balance check
counts = exp['group'].value_counts()
print("Group Balance:\n", counts)

# Conversion statistics
summary = exp.groupby('group')['converted'].agg(['count', 'sum', 'mean']).rename(columns={'mean': 'conversion_rate'})
conv_a = summary.loc['A', 'conversion_rate']
conv_b = summary.loc['B', 'conversion_rate']
relative_lift = ((conv_b - conv_a) / conv_a) * 100

print(f"\nGroup A Conversion Rate: {conv_a:.4f}")
print(f"Group B Conversion Rate: {conv_b:.4f}")
print(f"Relative Lift: {relative_lift:.2f}%")

# Contingency Table & Chi-Square Test
contingency = pd.crosstab(exp['group'], exp['converted'])
chi2, p_val, dof, ex = stats.chi2_contingency(contingency)
print(f"Chi-Square Statistic: {chi2:.4f}, p-value: {p_val:.4e}")

# 95% Confidence Interval for Difference in Proportions
n_a, n_b = summary.loc['A', 'count'], summary.loc['B', 'count']
diff = conv_b - conv_a
se = np.sqrt((conv_a * (1 - conv_a) / n_a) + (conv_b * (1 - conv_b) / n_b))
ci_lower = diff - (1.96 * se)
ci_upper = diff + (1.96 * se)

print(f"95% Confidence Interval for Difference: [{ci_lower:.4f}, {ci_upper:.4f}]")

### Statistical Interpretation
Because the $p$-value is below 0.05 ($p < 0.01$), we reject the null hypothesis $H_0$. The simplified sign-up screen (Group B) demonstrates a statistically significant uplift in onboarding conversions.